# YOLO Geometry

Decode one raw grid row, assign a target, and inspect NMS without a model dependency.

In [ ]:
from pathlib import Path
import importlib.util
lesson_rel = Path('phases/04-computer-vision/06-object-detection-yolo')
candidates = []
for start in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([start / lesson_rel / 'code/main.py', start / 'code/main.py'])
code_path = next(path.resolve() for path in candidates if path.is_file())
spec = importlib.util.spec_from_file_location('cv04_l06', code_path)
yolo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(yolo)
print(code_path)

In [ ]:
import numpy as np
anchors = [(16, 24), (32, 48)]
box = np.array([18, 20, 50, 68], dtype=float)
encoded = yolo.encode(box, 1, 1, 32, anchors[1])
decoded = yolo.decode(encoded, 1, 1, 32, anchors[1])
print('iou', yolo.box_iou(box[None], box[None])[0,0], 'encoded', encoded.round(3), 'error', np.abs(decoded-box).max())

In [ ]:
target, mask = yolo.assign_targets([box], [1], anchors, 32, (2, 2), 3)
total, parts = yolo.yolo_loss(np.zeros_like(target), target, mask)
raw = np.zeros((1, 2, 2, 2, 8), dtype=float)
raw[0, 0, 0, 0, 4] = 6
raw[0, 0, 0, 0, 6] = 6
print('target', target.shape, 'positive', int(mask.sum()), 'loss', total, parts)
print('detections', yolo.postprocess(raw, anchors, 32, conf_threshold=0.8)[0].shape)